# dk EGAT 모델 풀 학습 (Colab A100)

> **최종 업데이트**: 2026-07-14 (그래프/pair representation 고도화): ATLoss 교체 효과가 실측으로
> 확인됨(distant 프리트레인만 dev F1 46.58/Ign 44.21 — BCE 때 24.77에서 회복, 20k 기준 참고치
> 43.15~47.79 범위 안) → 보류해뒀던 고도화 2건 반영. **명령줄 인자 변경 없음** — 셀 3 커맨드는
> 그대로, 코드가 바뀌었을 뿐이라 재실행만 하면 됨.
>
> ① **Entity-Sentence Heterogeneous Graph**: 그래프 노드에 문장(Sentence)을 추가, entity-entity
> 엣지에 더해 entity-sentence("등장함") 엣지 신설. 직접/같은 문장/멘션겹침으로 안 이어진 두
> 엔티티도 공통 문장 노드를 거쳐 2-hop 만에 정보 교환 가능 (예: Steve Jobs -[S1]- Apple -[S2]-
> California). GAT 통과 후 엔티티 행만 잘라내 pair 구성에 사용.
>
> ② **Pair Representation에 element-wise interaction 추가**: `[g_h ‖ g_t ‖ c_ht]`(2304-d)에
> `g_h*g_t`(768-d) 곱 항을 더해 `[g_h ‖ g_t ‖ g_h*g_t ‖ c_ht]`(3072-d)로 확장.
>
> ③ **best-epoch 체크포인트 저장 추가**: 매 epoch dev F1이 갱신될 때마다 `{run_name}_best.pt`로
> 저장 — 마지막 epoch이 우연히 저점일 때(실측: epoch 6 F1 59.85 → epoch 7 59.57 하락 관측) 최고
> 기록을 놓치지 않기 위함. 최종 epoch 체크포인트(`{run_name}.pt`, baseline과 비교 기준)는 별도 유지.
>
> 이전: 손실함수를 BCE+threshold sweep → Adaptive Thresholding(ATLoss/PUATLoss)으로 교체 (distant
> 단계 PUATLoss na_weight=0.7, annotated는 자동 전환). Heterogeneous 그래프/interaction term은
> 그 당시엔 "손실함수 하나만 고쳐서 회복되는지 먼저 확인" 원칙으로 보류했다가, 회복이 확인되어
> 이번에 반영. 파이프라인 상세는 `Scripts/dk_gat/README.md`.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: distant 20,000개 × **1 epoch** PUATLoss(na_weight=0.7) 프리트레인 → annotated
3,053개 × **15 epoch** ATLoss 파인튜닝 (AdamW 2e-5, wd 0.01, warmup 6%, clip 1.0, seed 66) —
**`Scripts/atlop` baseline과 distant_limit/distant_epochs/epochs/seed를 전부 동일하게 맞춰서
GAT 추가라는 아키텍처 변수 하나만 비교**함. 예측은 Adaptive Thresholding으로 결정(전역 threshold
없음, 페어마다 학습된 TH 클래스와 비교) — 더 이상 threshold sweep 불필요.

**비교 기준** (`results/comparison.md`, 전부 동일 스코어러·seed 66·distant 20k·distant_epochs 1):

| 모델 | annotated epochs | dev F1 | Ign F1 |
|---|---|---|---|
| ATLOP baseline | 15 | 61.71 | 59.86 |
| ATLOP + PUATLoss(0.7) | 15 | 62.06 | 60.16 |
| dk EGAT (엔티티 그래프, ATLoss 교체 직후) | 8/15까지 관측 | 59.85(peak) | 58.12(peak) |
| **dk EGAT + Heterogeneous graph + interaction (이 실행)** | 15 | | |

In [19]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-bfacc661-c2ed-6655-72b8-12c6f6a3ac8d)


In [20]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# json들이 MB~GB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

Cloning into 'Information_Extraction'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 242 (delta 100), reused 167 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (242/242), 18.32 MiB | 13.03 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/Information_Extraction/Information_Extraction/Information_Extraction/Information_Extraction
Branch 'dk' set up to track remote branch 'dk' from 'origin'.
Switched to a new branch 'dk'
-rw-r--r-- 1 root root 4.1M Jul 14 04:36 docred_data/data/dev.json
-rw-r--r-- 1 root root 2.4K Jul 14 04:36 docred_data/data/rel_info.json
-rw-r--r-- 1 root root  132 Jul 14 04:36 docred_data/data/test.json
-rw-r--r-- 1 root root  13M Jul 14 04:36 docred_data/data/train_annotated.json
-rw-r--r-- 1 root root 417M Jul 14 04:39 docred_data/data/train_distant.json
colab_gat_a100.ipynb  model.py		 README.md
__init__.py	      preprocess_gat.py  t

In [21]:
# 2) 패키지
!pip install -q transformers==4.57.6 accelerate

In [22]:
# 3) 학습: distant 20k×1ep -> annotated 15ep (baseline과 동일 스케줄, A100 약 2~3시간)
#    epoch마다 [스테이지 | epoch N] dev_F1=.. Ign_F1=.. 로그가 찍힘
#    --use_pu_loss --na_weight 0.7: distant 단계만 PUATLoss, annotated는 자동으로 일반 ATLoss
#    (BCE+threshold sweep은 폐기 -- 실측 dev F1 24.77로 같은 20k 기준 RoBERTa+ATLoss의
#    43.15보다도 낮아서 Adaptive Thresholding으로 교체함, model.py 모듈 docstring 참고)
!python -m Scripts.dk_gat.train_gat \
  --distant_limit 20000 --distant_epochs 1 --epochs 15 \
  --use_pu_loss --na_weight 0.7 \
  --lr 2e-5 --weight_decay 0.01 --warmup_ratio 0.06 --dropout 0.1 \
  --run_name dk_gat --save_model --seed 66

[device] cuda
[data] train=3053 dev=998 docs
preprocess-gat: 100% 998/998 [00:07<00:00, 124.91it/s]
preprocess-gat: 100% 3053/3053 [00:24<00:00, 126.34it/s]
2026-07-14 04:44:59.518397: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 04:44:59.590641: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[stage 1] loss = PUATLoss(na_weight=0.7)
[stage 1] distant pretrain on 20000 docs (1 epoch(s))
preprocess-gat: 100% 20000/20000 [02:35<00:00, 128.69it/s]
  [distant-pretrain] epoch 0 step 50/5000 loss 2.6893
  [distan

In [23]:
# 4) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    best_* = dev F1 최고 epoch 체크포인트/예측 (마지막 epoch보다 나을 수 있음, 둘 다 백업)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat.pt results/dk_gat_best.pt \
   results/dk_gat_stage1.pt results/dk_gat_stage1_best.pt \
   results/dk_gat_dev_predictions.json results/dk_gat_best_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 773K Jul 14 05:49 dk_gat_best_dev_predictions.json
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_best.pt
-rw------- 1 root root 759K Jul 14 05:49 dk_gat_dev_predictions.json
-rw------- 1 root root 430M Jul 14 05:49 dk_gat.pt
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_stage1_best.pt
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_stage1.pt


## 결과 기록

마지막 epoch 로그 수치를 위의 비교표와 `results/comparison.md`에 반영. 로그에 `<- new best`가
찍힌 마지막 줄이 `dk_gat_best.pt`에 저장된 epoch — 최종 epoch 수치와 다르면 (더 낮으면) best
쪽 수치도 같이 기록해둘 것.

- stage-1(distant 직후) 수치도 따로 적어두면 GAT가 노이즈 라벨 위에서 얼마나 버티는지,
  annotated 파인튜닝이 얼마나 끌어올리는지 분해해 보여줄 수 있음.
- seed 66 단일 시드 — baseline과의 차이가 ±1점 이내면 PRD §5 시드 노이즈 주의 적용.
- Evidence Contrastive Loss는 annotated 단계에서만 실질 작동함(distant는 evidence 없음).
- GAT 고도화 추가 실험(edge feature ablation, 헤드 수, 인접행렬 반경 등)은 이 노트북의 셀 3에
  `--gat_heads`, `--evidence_weight` 인자만 바꿔 반복하면 됨.
- Meta-path별 attention 파라미터 분리, Curriculum PU-weight 스케줄은 이번에도 미반영 —
  이번 결과 확인 후 필요성 판단해 별도 실험으로 검토.